In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2
import matplotlib.patches as patches

join = os.path.join
from skimage import io
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as F
import torchvision
import monai
from monai.networks import one_hot
from segment_anything import SamPredictor, sam_model_registry
from segment_anything.utils.transforms import ResizeLongestSide
from utils.SurfaceDice import compute_dice_coefficient
# set seeds
torch.manual_seed(2024)
np.random.seed(2024)

In [ ]:
import json

# Opening JSON file
with open('../../../DATA/Amateur Drawing Semantic Segmentations/20240716-1034 (1)/label_definition.json') as json_file:
    data = json.load(json_file)

data['label_name_to_id']['CONFLICT']=26
data['label_name_to_id']['Conflicted']=26
data['label_name_to_id']['Unlabeled']=26

color_dict = {}
for label_name in data['label_name_to_color']:
    color_dict[int(data['label_name_to_id'][label_name])] = data['label_name_to_color'][label_name]

def labels_to_colors(img):
    w,h = img.shape[:2]
    img_rgb = np.zeros((w,h,3)).astype('uint8')
    for label_id in color_dict:
        img_rgb[img==label_id] = color_dict[label_id]
    return img_rgb


def img_to_label_id(img, bgr_format=False):
    if bgr_format:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    img = img.reshape(-1,3)
    b = img[:,0]
    g = img[:,1]
    r = img[:,2]
    labels = np.zeros_like(r)
    for classname in data['label_name_to_color']:
        color = data['label_name_to_color'][classname]
        mask = (r==color[0]) & (g==color[1]) & (b==color[2])
        labels[mask] = int(data['label_name_to_id'][classname])
    labels = labels.reshape(1024,1024)
    return labels

In [ ]:
# %% set up model for embedding computation

work_dir = './work_dir'
model_type = 'vit_b'
checkpoint = 'work_dir/SAM/sam_vit_b_01ec64.pth'
device = 'cuda:0'
num_classes = 27
sam_embedding_model = sam_model_registry[model_type](num_classes = num_classes, checkpoint=checkpoint).to(device)

In [ ]:
# %% set up model for segmentation inference

work_dir = './work_dir'
task_name = 'AutoSegDemo2D_300_no_binmask'
model_save_path = join(work_dir, task_name)
model_type = 'vit_b'
checkpoint = join(model_save_path, 'sam_model_best.pth')
device = 'cuda:0'
num_classes = 27
sam_model = sam_model_registry[model_type](num_classes = num_classes, checkpoint=checkpoint).to(device)

In [ ]:
# Input image
# image_data = io.imread("D:\\DATA\\Amateur Drawing Semantic Segmentations\\20240716-1034 (1)\\drawings_resized\\6d4b2a95a56741ada563c3f8cb8610c7.png")
# image_data = io.imread("D:\\GENERATION\\Animated SDXL\\image (5).png")
image_data = io.imread("D:\\GENERATION\\ANIM_SEG\\8_1.png")
if image_data.shape[-1]>3 and len(image_data.shape)==3:
    image_data = image_data[:,:,:3]
if len(image_data.shape)==2:
    image_data = np.repeat(image_data[:,:,None], 3, axis=-1)
# image_data = image_data[100:,200:]
image_data = cv2.resize(image_data,(1024,1024))
plt.imshow(image_data)
plt.axis("off")
plt.show()

In [ ]:
# if mask is available
# gt2D = io.imread("D:\\DATA\\Amateur Drawing Semantic Segmentations\\20240716-1034 (1)\\labels_resized_ID\\6d4b2a95a56741ada563c3f8cb8610c7__846691174187041.png")
# gt2D = io.imread("D:\\GENERATION\\Animated SDXL\\mask (5).png")
gt2D = io.imread("D:\\GENERATION\\ANIM_SEG\\8_gt.png")
if gt2D.shape[-1]==3:
    gt2D = img_to_label_id(gt2D, bgr_format=True)

# gt2D = gt2D[100:,200:]
gt2D = cv2.resize(gt2D,(1024,1024))

gt2D = torch.tensor(gt2D[None, :,:]).long()
THRESH = 0
binmask = gt2D[0]>THRESH
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3))
binmask = cv2.morphologyEx(binmask.numpy().astype('uint8'),cv2.MORPH_CLOSE,kernel)

# img_gray = cv2.cvtColor(image_data, cv2.COLOR_BGR2GRAY)
# binmask = img_gray<128
# gt2D = torch.tensor(binmask[None, :,:]).long()

plt.imshow(binmask, 'gray')
plt.axis("off")
plt.show()

In [ ]:
# resize image to 3*1024*1024
sam_transform = ResizeLongestSide(sam_model.image_encoder.img_size)
resize_img = sam_transform.apply_image(image_data)
resize_img_tensor = torch.as_tensor(resize_img.transpose(2, 0, 1)).to(device)
input_image = sam_model.preprocess(resize_img_tensor[None,:,:,:]) # (1, 3, 1024, 1024)
assert input_image.shape == (1, 3, sam_model.image_encoder.img_size, sam_model.image_encoder.img_size), 'input image should be resized to 1024*1024'

# preprocess binary mask
mask = F.resize(torch.Tensor(binmask).unsqueeze(0), 256, torchvision.transforms.InterpolationMode.NEAREST)
bg_mask = mask.unsqueeze(0)
bg_mask = bg_mask.float()
bg_mask = torch.FloatTensor(bg_mask).to(device)

# get bounding box from binary mask
Xs = np.where(binmask>0)[0] 
Ys = np.where(binmask>0)[1]
bbox = torch.from_numpy(np.array([min(Ys),min(Xs),max(Ys),max(Xs)])).float()
bbox = bbox//4 # resizing by a factor of 4 (1024-->256)
bbox = bbox.unsqueeze(0)
bbox = bbox.to(device)

with torch.no_grad():
    # pre-compute the image embedding
    ts_img_embedding = sam_embedding_model.image_encoder(input_image)

    
    sparse_embeddings, dense_embeddings = sam_model.prompt_encoder(
        points=None,
        boxes=bbox[:, None, :], # (B, 4) -> (B, 1, 4)
        masks=None,
    )
    medsam_seg_prob, _ = sam_model.mask_decoder(
        image_embeddings=ts_img_embedding.to(device), # (B, 256, 64, 64)
        image_pe=sam_model.prompt_encoder.get_dense_pe(), # (1, 256, 64, 64)
        sparse_prompt_embeddings=sparse_embeddings, # (B, 2, 256)
        dense_prompt_embeddings=dense_embeddings, # (B, 256, 64, 64)
        multimask_output=True,
        )
    medsam_seg_prob = torch.sigmoid(medsam_seg_prob)
    # convert soft mask to hard mask
    medsam_seg_prob = medsam_seg_prob.cpu().numpy().squeeze()
    medsam_seg = (medsam_seg_prob > 0.5).astype(np.uint8)

medsam_out = np.transpose(medsam_seg,(1,2,0))
medsam_out = cv2.resize(medsam_out, (1024,1024), cv2.INTER_LINEAR)
labels_out = torch.argmax(torch.Tensor(medsam_out), dim=2)

fig, ax = plt.subplots(1,4)
fig.set_size_inches(20,80)
ax[0].imshow(image_data)
ax[1].imshow(binmask, 'gray')
ax[2].imshow(labels_to_colors(labels_out.numpy().astype('uint8')))
ax[3].imshow(labels_to_colors(gt2D[0]))
bbox = bbox.cpu().numpy()[0]
rect = patches.Rectangle((bbox[0], bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1], linewidth=1, edgecolor='r', facecolor='none')
# ax.add_patch(rect)
ax[0].axis("off")
ax[1].axis("off")
ax[2].axis("off")
ax[3].axis("off")
plt.show()